# TP2: Compete on ML-Arena - 1 Minute Permuted MNIST

**SCAI-Sprint** | **Instructor:** Raphael Cousin

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/sprint/blob/main/tp2_competition.ipynb)

---

## Objective

In this practical, you will:
1. **Understand** the Permuted MNIST competition and the Agent interface
2. **Explore & train** agents locally in this notebook
3. **Deploy** your best agent on [ML-Arena](https://ml-arena.com) and climb the leaderboard

> **Prerequisite:** TP1 (bonus) covers PyTorch fundamentals — gradient descent, training loops, neural networks. Useful if you want to build a PyTorch-based agent.

---
# Part 1: Understanding the Competition

## What is "1 Minute Permuted MNIST"?

This is a **meta-learning** competition on [ML-Arena](https://ml-arena.com). The rules are simple:

- Your agent receives a series of **tasks** (episodes)
- Each task is a version of MNIST where **pixels are shuffled** and **labels are permuted**
- Your agent must **train** on the task's training data and **predict** on the test data
- **Constraint:** each task must complete (train + predict) in **less than 1 minute**
- Your score = **mean accuracy** across all tasks

### Why is it hard?

Each task has a **different** random permutation of pixels and labels. Your agent cannot memorize patterns from previous tasks — it must learn to adapt quickly from scratch every time.

### The Agent Interface

Your agent must be a Python class called `Agent` in a file called `agent.py` with two methods:

```python
class Agent:
    def train(self, X_train, y_train):
        """Learn from training data"""
        # X_train: numpy array of shape (60000, 28, 28) — images
        # y_train: numpy array of shape (60000, 1) — labels
        pass

    def predict(self, X_test) -> np.ndarray:
        """Predict labels for test data"""
        # X_test: numpy array of shape (10000, 28, 28) — images
        # Returns: numpy array of shape (10000,) — predicted labels
        pass
```

That's it! Everything happens inside these two methods.

---
# Part 2: Explore & Train Locally

## 1. Setup

In [ ]:
# Install the environment and clone agent examples
!pip install -q git+https://github.com/racousin/permuted_mnist.git
!git clone -q https://github.com/racousin/sprint.git /tmp/sprint 2>/dev/null || true

In [ ]:
import numpy as np
import time
import sys
import matplotlib.pyplot as plt

# Import the environment
from permuted_mnist.env.permuted_mnist import PermutedMNISTEnv

# Import the example agents from the repo
sys.path.insert(0, '/tmp/sprint/agents')
from random_agent import Agent as RandomAgent
from linear_agent import Agent as LinearAgent

print("Setup complete!")

## 2. Explore the Data

Let's see what a permuted MNIST task looks like:

In [ ]:
# Create the environment
env = PermutedMNISTEnv(number_episodes=10)
env.set_seed(42)

# Get the first task
task = env.get_next_task()

print("Task structure:")
print(f"  X_train: {task['X_train'].shape}  — 60,000 images of 28x28 pixels")
print(f"  y_train: {task['y_train'].shape}  — 60,000 labels")
print(f"  X_test:  {task['X_test'].shape}  — 10,000 images")
print(f"  y_test:  {task['y_test'].shape}   — 10,000 labels (for local evaluation)")

In [ ]:
# Visualize some permuted images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Permuted MNIST — pixels and labels are shuffled!', fontsize=14)

for i in range(10):
    ax = axes[i // 5, i % 5]
    ax.imshow(task['X_train'][i], cmap='gray')
    ax.set_title(f'Label: {task["y_train"][i][0]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("The images look scrambled — that's the pixel permutation!")
print("The labels are also permuted (a '3' might be labeled '7').")

## 3. Baseline: Random Agent (~10% accuracy)

Let's start with the simplest possible agent — random predictions.

Take a look at the code in `agents/random_agent.py`:

In [ ]:
# Show the random agent code
!cat /tmp/sprint/agents/random_agent.py

In [ ]:
# Evaluate the random agent
env.reset()
env.set_seed(42)

random_agent = RandomAgent(output_dim=10, seed=42)
random_accuracies = []
random_times = []

print("Random Agent")
print("=" * 50)

task_num = 1
while True:
    task = env.get_next_task()
    if task is None:
        break

    start = time.time()
    random_agent.train(task['X_train'], task['y_train'])
    preds = random_agent.predict(task['X_test'])
    elapsed = time.time() - start

    acc = env.evaluate(preds, task['y_test'])
    random_accuracies.append(acc)
    random_times.append(elapsed)
    print(f"  Task {task_num}: {acc:.2%} accuracy, {elapsed:.4f}s")
    task_num += 1

print(f"\nMean accuracy: {np.mean(random_accuracies):.2%}")
print("As expected — random guessing on 10 classes gives ~10%.")

## 4. Linear Agent (~91% accuracy)

Now a simple linear classifier (logistic regression with SGD).

Take a look at the code in `agents/linear_agent.py`:

In [ ]:
# Show the linear agent code
!cat /tmp/sprint/agents/linear_agent.py

In [ ]:
# Evaluate the linear agent
env.reset()
env.set_seed(42)

linear_agent = LinearAgent(input_dim=784, output_dim=10, learning_rate=0.01)
linear_accuracies = []
linear_times = []

print("Linear Agent")
print("=" * 50)

task_num = 1
while True:
    task = env.get_next_task()
    if task is None:
        break

    linear_agent.reset()
    start = time.time()
    linear_agent.train(task['X_train'], task['y_train'], epochs=5, batch_size=32)
    preds = linear_agent.predict(task['X_test'])
    elapsed = time.time() - start

    acc = env.evaluate(preds, task['y_test'])
    linear_accuracies.append(acc)
    linear_times.append(elapsed)
    print(f"  Task {task_num}: {acc:.2%} accuracy, {elapsed:.2f}s")
    task_num += 1

print(f"\nMean accuracy: {np.mean(linear_accuracies):.2%}")
print(f"Total time: {np.sum(linear_times):.1f}s")

## 5. Compare Results

In [ ]:
# Comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

tasks = np.arange(1, len(random_accuracies) + 1)

# Accuracy
ax1.plot(tasks, random_accuracies, 'o-', label='Random (~10%)', alpha=0.7)
ax1.plot(tasks, linear_accuracies, 's-', label='Linear (~91%)', alpha=0.7)
ax1.axhline(y=0.1, color='gray', linestyle='--', alpha=0.5, label='Random chance')
ax1.set_xlabel('Task')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy per Task')
ax1.legend()
ax1.set_ylim([0, 1])
ax1.grid(True, alpha=0.3)

# Time
ax2.bar(tasks - 0.2, random_times, 0.4, label='Random', alpha=0.7)
ax2.bar(tasks + 0.2, linear_times, 0.4, label='Linear', alpha=0.7)
ax2.axhline(y=60, color='red', linestyle='--', alpha=0.5, label='1 min limit')
ax2.set_xlabel('Task')
ax2.set_ylabel('Time (s)')
ax2.set_title('Time per Task')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Random Agent:  {np.mean(random_accuracies):.2%} mean accuracy")
print(f"Linear Agent:  {np.mean(linear_accuracies):.2%} mean accuracy")
print(f"\nCan you do better? That's the challenge!")

## 6. Exercise: Build Your Own Agent

Now it's your turn! Use what you learned in TP1 to build a better agent.

**Ideas to try:**
- A neural network with hidden layers (using PyTorch or NumPy)
- Better hyperparameters (learning rate, epochs, batch size)
- Data preprocessing (normalization, flattening strategies)
- Early stopping to save time

**Remember the constraints:**
- Your agent must complete each task in **< 1 minute**
- It must implement `train(X_train, y_train)` and `predict(X_test)`
- `predict` must return a numpy array of integer labels

In [ ]:
# YOUR AGENT — modify this!

class MyAgent:
    def __init__(self):
        # TODO: initialize your model
        pass

    def reset(self):
        # TODO: reset for a new task
        pass

    def train(self, X_train, y_train):
        # TODO: train on the task data
        # X_train: (60000, 28, 28), y_train: (60000, 1)
        pass

    def predict(self, X_test):
        # TODO: return predicted labels
        # X_test: (10000, 28, 28) -> return (10000,) array of ints
        n_samples = X_test.shape[0]
        return np.zeros(n_samples, dtype=int)  # Replace this!

In [ ]:
# Test your agent
env.reset()
env.set_seed(42)

my_agent = MyAgent()
my_accuracies = []
my_times = []

print("Your Agent")
print("=" * 50)

task_num = 1
while True:
    task = env.get_next_task()
    if task is None:
        break

    my_agent.reset()
    start = time.time()
    my_agent.train(task['X_train'], task['y_train'])
    preds = my_agent.predict(task['X_test'])
    elapsed = time.time() - start

    acc = env.evaluate(preds, task['y_test'])
    my_accuracies.append(acc)
    my_times.append(elapsed)

    status = "OK" if elapsed < 60 else "TOO SLOW!"
    print(f"  Task {task_num}: {acc:.2%} accuracy, {elapsed:.2f}s [{status}]")
    task_num += 1

print(f"\nMean accuracy: {np.mean(my_accuracies):.2%}")
print(f"Max time per task: {np.max(my_times):.2f}s {'(< 1 min)' if np.max(my_times) < 60 else '(OVER LIMIT!)'}")

if np.mean(my_accuracies) > np.mean(linear_accuracies):
    print(f"\nYou beat the linear agent! Ready to deploy.")
else:
    print(f"\nLinear agent still wins ({np.mean(linear_accuracies):.2%}). Keep iterating!")

---
# Part 3: Deploy on ML-Arena

Once you're happy with your agent's performance locally, it's time to deploy it on the competition platform!

## Step 1: Enroll in the Competition

Go to the enrollment link and create your account:

**https://ml-arena.com/enroll/6994826ca2716b19f19147e637879ccc**

This will bring you to the **"1 Minute Permuted MNIST"** competition page.

## Step 2: Create Your Agent

1. Click **"Submit Agent"** on the competition page
2. **Name your agent** (pick any name you like)
3. Click **"Next step"**

## Step 3: Upload Your Code

1. Upload your `agent.py` file — this is the file containing your `Agent` class
2. Configure the **Agent Runtime**:
   - **Language:** Python
   - **Version:** 3.10
   - **Framework:** PyTorch (if your agent uses it, otherwise select "none")
3. If your agent has additional dependencies, add them in `requirements.txt`
4. Click **"Proceed to Deployment"**

## Step 4: Deploy

1. The platform will **validate** your upload (check that agent.py has the right interface)
2. If validation passes, click **"Deploy Agent"**
3. The deployment process runs:
   - **2 dry-run tests** to validate performance
   - **10 competitive runs** for initial ranking
4. You have **200 deployments per day** — iterate freely!

## Step 5: Check the Leaderboard

Go to the **Leaderboard** tab on the competition page to see your ranking.

Your score is the **mean accuracy** across all competitive runs. Keep improving your agent and redeploying to climb the leaderboard!

---
# Part 4: Prepare Your `agent.py` for Upload

Before uploading, make sure your agent file is self-contained.

Here's the template:

In [ ]:
# Save your agent to a file for upload
# Modify the code below with your best agent, then run this cell to save it

agent_code = '''
import numpy as np


class Agent:
    def __init__(self):
        # Initialize your model here
        self.input_dim = 784
        self.output_dim = 10
        self.learning_rate = 0.01
        self.reset()

    def reset(self):
        # Reset weights for a new task
        self.W = np.random.randn(self.input_dim, self.output_dim) * np.sqrt(2.0 / self.input_dim)
        self.b = np.zeros((1, self.output_dim))

    def train(self, X_train, y_train):
        # TODO: Replace with your training logic
        pass

    def predict(self, X_test):
        # TODO: Replace with your prediction logic
        n_samples = X_test.shape[0]
        return np.zeros(n_samples, dtype=int)
'''

with open('agent.py', 'w') as f:
    f.write(agent_code.strip())

print("agent.py saved! Download it and upload to ML-Arena.")
print("In Colab: click the folder icon on the left > right-click agent.py > Download")

---
# Summary

| Agent | Accuracy | Time/Task |
|-------|----------|----------|
| Random baseline | ~10% | instant |
| Linear (logistic regression) | ~91% | ~1.4s |
| **Your agent** | **???** | **< 60s** |

### Tips for improving your score:
- **Neural networks** with hidden layers can capture non-linear patterns
- **Learning rate tuning** — try 0.1, the linear agent improves from 91% to 92%
- **More epochs** — but watch the 1-minute time limit
- **PyTorch** — use GPU acceleration if available (select PyTorch framework on ML-Arena)
- **Data augmentation** or **regularization** to prevent overfitting

Good luck on the leaderboard!